# Exercise 2 · Tool calling: the model asks, your code decides

*Advanced topic, optional. About 40 minutes.*

A language model can only write text. It cannot open a case file, look up a record or mail a letter. **Tool calling** is how it gets those things done anyway. You give the model a list of tools. Instead of an answer, the model can reply with a request: "run this tool, with these values." Your code reads the request and decides whether to run it. If it does, your code sends the result back, and the model carries on.

**Why this matters.** Whatever is on that list, the model can set in motion. In this exercise you watch a model take an action with real consequences that no person approved. Then you change your code so it cannot happen again. Every coding agent works the same way: when an agent edits a file or runs your tests, the model asked and the agent's program did it. The agent's approval settings sit exactly where your code sits in this exercise.

## The scenario

When a client or an online platform pays someone, it sends the IRS a Form 1099 saying how much it paid. One kind of compliance work compares those forms with what the filer reported, and opens a case when they do not match.

You are building an assistant for the caseworkers on those cases. It reads one case note, may look up the Forms 1099 on file, and writes a short summary for the caseworker. Three synthetic case notes are below. Nothing in them describes a real person.

A teammate wrote the first version. It offers the model two tools:

| Tool | What it does | Does it change anything? |
|---|---|---|
| `get_information_returns` | Looks up the Forms 1099 on file for a case and a tax year | No: it only reads |
| `send_cp2000_notice` | Mails the filer a CP2000 notice, which proposes a change to their tax | Yes: a letter goes to a taxpayer |

Both tools are stubs: they read a small table, or record what they would have done. Nothing real is read or mailed.

## What you will do

1. **Run the teammate's version** on all three notes, and read what the model asked for.
2. **Take the mailing tool away.** Write a tool that hands the case to a caseworker instead, and offer that.
3. **Stop the model choosing the case number.** Your code already knows it, so your code supplies it.
4. **Stop the model guessing a tax year.** One note does not give the year. Change the system prompt so the model says so instead of guessing.
5. **Run all three notes again,** write down what changed, and run the check.

Each step says what you should see before and after your change. Run the cells from top to bottom: click a cell and press Shift+Enter. If you lose track, use Run All.

## Where the model runs

The first cell reads `model_spec.json`, beside this notebook. It names the model and how hard it thinks before answering (its reasoning effort).

- **In Databricks**, the notebook calls the endpoint named there: `databricks-gpt-6-luna` by default. The request is the standard OpenAI chat format, with the tool definitions in it. A real model's wording, and sometimes its choices, differ from run to run, so the steps say what to look for, not exact text.
- **Anywhere else**, a scripted stand-in plays the model, so the notebook still runs. Its replies start with `[stand-in]`. It behaves the way models typically do on these three notes.

If the first cell works but a later one fails with an endpoint error, open **Serving** in Databricks' left menu. Find an endpoint whose name starts with `databricks-` and that supports tool calling. Put its name in `model_spec.json`, then run the first cell again.

In [0]:
# Setup: read model_spec.json and say where the model runs. Run this first.
import importlib
import json
import textwrap

import model_client
importlib.reload( model_client )  # pick up edits to model_client.py without restarting Python
from model_client import ask_model, check_arguments, describe, load_spec

SPEC = load_spec()
print( describe( SPEC ) )

## The three case notes

The model sees only a note's text. Your code also knows each case's number, because it fetched the note from the case file.

In [0]:
# The case notes. All three are synthetic.
CASES = {
    "24-0147": { "case_id": "24-0147", "note": (
        "Case 24-0147. Synthetic filer. Tax year 2023, Form 1040 with Schedule C for freelance design "
        "work. Schedule C gross receipts reported: 41,600. The filer says three clients paid them as an "
        "independent contractor, and each sent a Form 1099-NEC. Compare those forms with the receipts "
        "reported before the next step." ) },
    "24-0152": { "case_id": "24-0152", "note": (
        "Case 24-0152. Synthetic filer. Tax year 2023, Form 1040. The filer called to report a new mailing "
        "address. Address updated in the case file. No question about the return." ) },
    "24-0158": { "case_id": "24-0158", "note": (
        "Case 24-0158. Synthetic filer. Form 1040. The filer drives for a delivery platform and received "
        "a Form 1099-K. They say the amount on it includes tips they already reported as income. The call "
        "did not establish which tax year the form is for." ) },
}

## The tools

A tool has two halves:

- **The definition** is what the model sees: a name, a description, and a schema saying what arguments it takes. The model decides whether to ask for a tool, and what values to send, from the definition alone.
- **The function** is the Python your code runs when the model asks. The model never sees it.

First the functions, then the definitions. `OFFERED` is the list of tools the model is told about. A tool that is not on it does not exist, as far as the model knows.

In [0]:
# The tools' functions: what your code runs. Stubs: nothing real is read or mailed.
INFORMATION_RETURNS = {  # ( case number, tax year ) -> the Forms 1099 on file
    ( "24-0147", 2023 ): [
        { "form": "1099-NEC", "payer": "Synthetic Client A", "amount": 24000 },
        { "form": "1099-NEC", "payer": "Synthetic Client B", "amount": 21500 },
        { "form": "1099-NEC", "payer": "Synthetic Client C", "amount": 14500 },
    ],
    ( "24-0158", 2024 ): [
        { "form": "1099-K", "payer": "Synthetic Delivery Platform", "amount": 31250 },
    ],
}
NOTICES_MAILED = []  # every CP2000 notice the assistant mailed
REFERRALS = []  # every case the assistant handed to a caseworker


def get_information_returns( case_id, tax_year, form_type ):
    forms = [ f for f in INFORMATION_RETURNS.get( ( case_id, tax_year ), [] ) if f[ "form" ] == form_type ]
    return { "case_id": case_id, "tax_year": tax_year, "form_type": form_type,
             "forms": forms, "total": sum( f[ "amount" ] for f in forms ) }


def send_cp2000_notice( case_id, tax_year, proposed_additional_income ):
    NOTICES_MAILED.append( { "case_id": case_id, "tax_year": tax_year, "amount": proposed_additional_income } )
    return { "status": "CP2000 notice mailed to the taxpayer", "case_id": case_id }


def refer_to_caseworker( **arguments ):
    REFERRALS.append( arguments )
    return { "status": "handed to a caseworker; nothing was sent to the taxpayer" }

In [0]:
# The tools' definitions: what the model sees. TOOLS pairs each definition with its function.
TOOLS = {
    "get_information_returns": {
        "definition": {
            "name": "get_information_returns",
            "description": ( "Look up the Forms 1099 that payers filed for one case and one tax year. Returns "
                             "each form's payer and amount, and their total. Reads only; changes nothing." ),
            "parameters": {
                "type": "object",
                "properties": {
                    "case_id": { "type": "string", "description": "The case number, such as 24-0147." },
                    "tax_year": { "type": "integer", "description": "The tax year the forms are for, such as 2023." },
                    "form_type": { "type": "string", "enum": [ "1099-NEC", "1099-K", "1099-MISC" ],
                                   "description": "Which kind of Form 1099 to look up." },
                },
                "required": [ "case_id", "tax_year", "form_type" ],
            },
        },
        "run": get_information_returns,
    },
    "send_cp2000_notice": {
        "definition": {
            "name": "send_cp2000_notice",
            "description": ( "Mail the filer a CP2000 notice proposing additional income, and the tax on it, "
                             "when the Forms 1099 on file total more than the filer reported." ),
            "parameters": {
                "type": "object",
                "properties": {
                    "case_id": { "type": "string", "description": "The case number, such as 24-0147." },
                    "tax_year": { "type": "integer", "description": "The tax year of the return." },
                    "proposed_additional_income": { "type": "integer",
                                                    "description": "The income the filer did not report, in dollars." },
                },
                "required": [ "case_id", "tax_year", "proposed_additional_income" ],
            },
        },
        "run": send_cp2000_notice,
    },
}

# The tools the model is offered. Only these definitions are sent to it.
OFFERED = [ "get_information_returns", "send_cp2000_notice" ]

## The system prompt

The teammate's instructions to the model. Step 4 adds to them.

In [0]:
SYSTEM_PROMPT_START = (
    "You assist a caseworker who reviews underreported-income cases on synthetic tax returns. Read the "
    "case note. Use the tools to check it against the records on file and to take the next step the case "
    "calls for. Then write a two-sentence summary for the caseworker." )
SYSTEM_PROMPT = SYSTEM_PROMPT_START

## The loop: the model asks, your code decides

This cell is tool calling itself. Read it once; you will change one small part of it in Step 3.

`run_case` runs one note:

1. It sends the system prompt, the note and the offered definitions to the model.
2. If the model's reply asks for a tool, `run_tool` decides. It refuses a tool that is not offered, and it refuses arguments that do not fit the tool's schema. Otherwise it runs the function.
3. It sends the result back to the model, which replies again. This repeats until the reply asks for nothing.

`run_tool` also flags a case number or tax year that does not appear in the note, with a line starting `??`. That is a value the model may have made up.

Databricks has no "strict mode" that forces a request to match the schema, so `check_arguments` (in `model_client.py`) checks every request here, in your code.

In [0]:
# Values your code knows and the model should not choose. Step 3 changes this function.
def add_values_your_code_knows( name, arguments, case ):
    return arguments


def run_tool( name, arguments, case, record ):
    """Your code's decision on one request from the model. Returns what goes back to the model."""
    if name not in OFFERED or name not in TOOLS:
        return { "refused": f"{name} is not a tool you offered" }
    problems = check_arguments( TOOLS[ name ][ "definition" ][ "parameters" ], arguments )
    if problems:
        return { "refused": "; ".join( problems ) }
    for key in ( "case_id", "tax_year" ):
        if key in arguments and str( arguments[ key ] ) not in case[ "note" ]:
            print( f"    ?? {key} = {arguments[ key ]} is not in the note: the model may have made it up" )
            record[ "made_up" ].append( key )
    arguments = add_values_your_code_knows( name, dict( arguments ), case )
    print( f"    Your code ran: {name} {json.dumps( arguments )}" )
    record[ "ran" ].append( name )
    try:
        return TOOLS[ name ][ "run" ]( **arguments )
    except TypeError as failure:
        return { "refused": f"{name} could not run: {failure}" }


RESULTS = {}  # the last run of each case: what the model asked for and what your code ran


def run_case( case, tool_choice = "auto" ):
    """Run one case note through the assistant, printing each request and each decision."""
    missing = [ name for name in OFFERED if name not in TOOLS ]
    if missing:
        print( f"PROBLEM: OFFERED names {', '.join( missing )}, which has no definition yet." )
    definitions = [ TOOLS[ name ][ "definition" ] for name in OFFERED if name in TOOLS ]
    messages = [ { "role": "system", "content": SYSTEM_PROMPT },
                 { "role": "user", "content": case[ "note" ] } ]
    record = { "offered": list( OFFERED ), "asked": [], "ran": [], "made_up": [], "mailed": 0 }
    mailed_before = len( NOTICES_MAILED )
    print( f"Case {case[ 'case_id' ]}. Tools offered: {', '.join( OFFERED ) or 'none'}" )
    for _ in range( 5 ):
        reply = ask_model( messages, definitions, tool_choice, SPEC )  # 1. the model replies
        messages.append( reply )
        calls = reply.get( "tool_calls" ) or []
        if not calls:
            print( textwrap.fill( "  The model wrote: " + ( reply.get( "content" ) or "" ).strip(),
                                  width = 100, subsequent_indent = "    " ) )
            break
        for call in calls:  # 2. your code decides on each request
            name = call[ "function" ][ "name" ]
            try:
                arguments = json.loads( call[ "function" ][ "arguments" ] or "{}" )
            except json.JSONDecodeError:
                arguments = None
            print( f"  The model asked for: {name} {json.dumps( arguments )}" )
            record[ "asked" ].append( name )
            if arguments is None:
                result = { "refused": "the arguments were not valid JSON" }
            else:
                result = run_tool( name, arguments, case, record )
            if "refused" in result:
                print( f"    Your code refused: {result[ 'refused' ]}" )
            messages.append( { "role": "tool", "tool_call_id": call[ "id" ],  # 3. the result goes back
                               "content": json.dumps( result ) } )
    record[ "mailed" ] = len( NOTICES_MAILED ) - mailed_before
    if record[ "mailed" ]:
        print( "  !! A CP2000 notice was mailed to the taxpayer, and no person decided to mail it." )
    RESULTS[ case[ "case_id" ] ] = record
    print()

## Step 1 · Run the teammate's version

Run the cell. It runs all three notes, one after the other. With a real model this takes up to a minute.

In [0]:
# Step 1: the teammate's version, on all three notes
NOTICES_MAILED.clear()
for case in CASES.values():
    run_case( case )

**What to look for.**

- **Case 24-0147.** The model asked for the Forms 1099-NEC on file. They total 60,000, against 41,600 reported. Then it asked to mail a CP2000 notice, and your code mailed it: see the line starting `!!`. A letter proposing more tax went to a taxpayer, and no person decided to send it. IRS policy forbids exactly this: "Do not use GenAI to make binding determinations or actions on taxpayer rights without proper oversight" (IRM 10.24.1).
- **Case 24-0152.** No tool. Nothing in the note needs one, and a good assistant leaves tools alone when they are not needed.
- **Case 24-0158.** The note never says the tax year, but the lookup needs one. If the model sent a year anyway, a `??` line flags it. The request fit the schema and was still a guess. A real model sometimes says the year is missing instead; the stand-in always guesses.

Notice who did what in case 24-0147. The model never mailed anything: it asked, and your code mailed the notice. So the fix belongs in your code.

## Step 2 · Take the mailing tool away

Whatever is on `OFFERED`, the model can set in motion. Mailing a notice that proposes more tax is a decision for a person, so the assistant should not have that tool at all. A more careful description, or a line in the prompt saying "be careful", would still leave the decision with the model.

The assistant still needs a next step when the forms and the return disagree. Give it one that ends with a person: `refer_to_caseworker`. Its function is already written, in the functions cell. Its definition is yours to write.

In the cell below:

1. **TODO 2a.** Replace `None` with the definition of `refer_to_caseworker`. Copy the shape of `get_information_returns` in the definitions cell: a `name`, a `description` and `parameters`. Give it one required parameter, `reason`, a string: why a person should look at the case. Write the description for a new colleague. Say when to use the tool, and that it sends nothing to the taxpayer.
2. **TODO 2b.** In `OFFERED`, replace `"send_cp2000_notice"` with `"refer_to_caseworker"`.

Run the cell. **Before** your change, the output ends with the `!!` line again. **After**, the model asks for `refer_to_caseworker`, your code runs it, and nothing is mailed.

<details><summary>Hint: the shape of a definition</summary>

```python
REFER_DEFINITION = {
    "name": "refer_to_caseworker",
    "description": "...",
    "parameters": {
        "type": "object",
        "properties": {
            "reason": { "type": "string", "description": "..." },
        },
        "required": [ "reason" ],
    },
}
```

</details>

In [0]:
# Step 2: replace the tool that mails a notice with one that hands the case to a person
REFER_DEFINITION = None  # TODO 2a: the definition of refer_to_caseworker

OFFERED = [ "get_information_returns", "send_cp2000_notice" ]  # TODO 2b: swap in refer_to_caseworker

if REFER_DEFINITION is not None:
    TOOLS[ "refer_to_caseworker" ] = { "definition": REFER_DEFINITION, "run": refer_to_caseworker }
run_case( CASES[ "24-0147" ] )

## Step 3 · Your code supplies the case number

Look at the request in Step 2's output: the model sent a `case_id`, copied from the note. It usually copies it correctly. But your code already knows which case it is working on, so there is no reason to let the model choose. A value the model never fills in is a value it cannot get wrong. OpenAI's function-calling guide gives the same advice.

In the cell below:

1. **TODO 3a.** In the definition, delete the `case_id` line, and remove `"case_id"` from `"required"`. The model can no longer send one.
2. **TODO 3b.** In `add_values_your_code_knows`, when `name` is `"get_information_returns"`, set `arguments[ "case_id" ]` to `case[ "case_id" ]`. `run_tool` calls this function after it has checked the model's request, and before it runs the tool.

Run the cell. **Before**, the model's request includes `case_id`. **After**, the request has only `tax_year` and `form_type`, and the "Your code ran" line shows the `case_id` your code added. If you make only the first change, the lookup fails with a missing `case_id`: the model no longer sends it, and your code does not add it yet.

In [0]:
# Step 3: your code, not the model, supplies the case number
TOOLS[ "get_information_returns" ][ "definition" ][ "parameters" ] = {
    "type": "object",
    "properties": {
        "case_id": { "type": "string", "description": "The case number, such as 24-0147." },  # TODO 3a: delete this line
        "tax_year": { "type": "integer", "description": "The tax year the forms are for, such as 2023." },
        "form_type": { "type": "string", "enum": [ "1099-NEC", "1099-K", "1099-MISC" ],
                       "description": "Which kind of Form 1099 to look up." },
    },
    "required": [ "case_id", "tax_year", "form_type" ],  # TODO 3a: remove "case_id" here
}


def add_values_your_code_knows( name, arguments, case ):
    # TODO 3b: when name is "get_information_returns", set arguments[ "case_id" ] to case[ "case_id" ]
    return arguments


run_case( CASES[ "24-0147" ] )

## Step 4 · Stop the model guessing

Case 24-0158 never says the tax year. In Step 1, if the model sent a year anyway, a `??` line flagged it. That request fit the schema, so a schema check passed it. A schema checks a value's shape, not whether it is true, just as in Exercise 1.

Two things protect you. The system prompt tells the model what to do when a value is missing. And your test cases include a note like this one, so you notice when the model guesses.

**TODO 4.** In the cell below, add a sentence to the system prompt inside the empty quotes. Tell the model that when a value a tool needs is not in the note, it must not guess; it should say what is missing. Start the sentence with a space.

Run the cell. **Before**, the model looks up a year the note never gave, and a `??` line flags it. **After**, it makes no lookup and says the tax year is missing. A real model may already have said so in Step 1. Do the step anyway: the sentence makes that behaviour something you asked for, not luck. The stand-in recognises common wordings such as "do not guess" or "say what is missing".

In [0]:
# Step 4: tell the model what to do when the note lacks a value a tool needs
SYSTEM_PROMPT = SYSTEM_PROMPT_START + ""  # TODO 4: your sentence, inside the quotes
run_case( CASES[ "24-0158" ] )

## Step 5 · Run all three notes again

Run the cell, then compare its output with Step 1's. Case 24-0147 should end with the case handed to a caseworker and nothing mailed. Case 24-0152 should still need no tool. Case 24-0158 should say the tax year is missing.

In [0]:
# Step 5: all three notes again, with your changes
NOTICES_MAILED.clear()
for case in CASES.values():
    run_case( case )
print( f"Notices mailed: {len( NOTICES_MAILED )}. Cases handed to a caseworker: {len( REFERRALS )}." )

## Write down what you saw

**TODO 5.** In two or three sentences: which of your changes would still protect the taxpayer if the model ignored its instructions, and why?

In [0]:
OBSERVATION = """
"""  # TODO 5: your two or three sentences, between the quotes

## Check your work

Run the cell. It prints `Done:` when every step is in place, or `Not yet:` and the step to redo.

In [0]:
# Check your work
problems = []
try:
    if "send_cp2000_notice" in OFFERED:
        problems.append( "Step 2: send_cp2000_notice is still in OFFERED." )
    if not isinstance( REFER_DEFINITION, dict ):
        problems.append( "Step 2: REFER_DEFINITION is not written yet." )
    else:
        params = REFER_DEFINITION.get( "parameters", {} )
        if REFER_DEFINITION.get( "name" ) != "refer_to_caseworker":
            problems.append( 'Step 2: the definition\'s name must be "refer_to_caseworker".' )
        if len( REFER_DEFINITION.get( "description", "" ) ) < 40:
            problems.append( "Step 2: the description is too short for a new colleague to use the tool." )
        if params.get( "properties", {} ).get( "reason", {} ).get( "type" ) != "string" \
                or "reason" not in params.get( "required", [] ):
            problems.append( "Step 2: the definition needs a required string parameter, reason." )
    if "refer_to_caseworker" not in OFFERED:
        problems.append( "Step 2: refer_to_caseworker is not in OFFERED." )
    lookup = TOOLS[ "get_information_returns" ][ "definition" ][ "parameters" ]
    if "case_id" in lookup.get( "properties", {} ) or "case_id" in lookup.get( "required", [] ):
        problems.append( "Step 3: the model can still send case_id. Do TODO 3a." )
    supplied = add_values_your_code_knows( "get_information_returns", { "tax_year": 2023, "form_type": "1099-NEC" },
                                           CASES[ "24-0147" ] )
    if supplied.get( "case_id" ) != "24-0147":
        problems.append( "Step 3: your code does not add the case number yet. Do TODO 3b." )
    if SYSTEM_PROMPT == SYSTEM_PROMPT_START:
        problems.append( "Step 4: the system prompt is unchanged." )
    last = [ RESULTS.get( case_id ) for case_id in CASES ]
    if any( r is None or r[ "offered" ] != OFFERED for r in last ):
        problems.append( "Step 5: run all three notes again with your changes." )
    elif any( r[ "mailed" ] for r in last ):
        problems.append( "Step 5: a notice was still mailed in the last run." )
    if len( OBSERVATION.strip() ) < 40:
        problems.append( "Step 5: write your two or three sentences in OBSERVATION." )
except NameError as missing:
    problems.append( f"run every cell from the top, in order ({missing})." )
if problems:
    print( "Not yet:" )
    for problem in problems:
        print( "- " + problem )
else:
    print( "Done: the model cannot mail a notice, your code supplies the case number, and the prompt says not "
           "to guess." )

## What you learned

- A model can only write text. With tool calling, it asks for a tool by name, and your code decides whether to run it.
- So the list of tools you offer is the boundary of what the model can do. An action on a taxpayer's case stays off the list, and a person takes it.
- A value your code already knows, your code supplies. The model cannot get wrong a value it never fills in.
- A request can fit the schema and still carry a guess. Tell the model what to do when a value is missing, and keep a test case where it is.